# 16 Compute stock return features

目标：按自然月窗口计算 6 个历史收益特征，并执行 Kenneth French r12_2 纳入条件。

- 输入：14 号清洗月度股票数据
- 正式输出：stock_return_features_raw.parquet、r12_2 排除审计
- 主键：permno+month
- 上游依赖：14
- smoke 模式仅使用真实本地数据子样本，并写入 `data/audit/smoke/`；不冒充正式结果。

In [ ]:
from __future__ import annotations
import json, os, sys, time
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config_utils import load_project_config
from src.path_utils import find_project_root
from src.runtime_paths import (
    clean_output_path,
    execution_context,
    install_clean_read_guard,
)

ROOT = find_project_root(ROOT)
EXECUTION_CONTEXT = execution_context()
install_clean_read_guard(ROOT)
CONFIG = load_project_config(ROOT)
PARAMS = CONFIG["parameters"]
PATHS = CONFIG["resolved_paths"]
RUN_MODE = os.getenv("KANIEL_RUN_MODE", "smoke").strip().lower()
SMOKE = RUN_MODE != "full"
if RUN_MODE == "full":
    FULL_RUN_TMP = (
        clean_output_path(ROOT, "data/interim/_full_run_tmp")
        if EXECUTION_CONTEXT == "clean_rebuild_compute"
        else PATHS["full_run_tmp"]
    )
    PATHS["full_run_tmp"] = FULL_RUN_TMP
    FULL_RUN_TMP.mkdir(parents=True, exist_ok=True)
    os.environ["TMP"] = str(FULL_RUN_TMP)
    os.environ["TEMP"] = str(FULL_RUN_TMP)
    os.environ["TMPDIR"] = str(FULL_RUN_TMP)
    os.environ["POLARS_TEMP_DIR"] = str(FULL_RUN_TMP)
STARTED_AT = time.time()
print({
    "project_root": str(ROOT),
    "run_mode": RUN_MODE,
    "execution_context": EXECUTION_CONTEXT,
})


In [ ]:
import os
import duckdb
from src.atomic_write import duckdb_copy_csv_atomic, duckdb_copy_parquet_atomic
from src.runtime_paths import output_path, upstream_path

def esc(path): return str(path).replace("'","''")
monthly_path=upstream_path(ROOT,RUN_MODE,"14_prepare_crsp_stock_monthly_daily","data/processed/stock/stock_monthly_clean.parquet")
con=duckdb.connect()
con.execute(f"SET memory_limit='{PARAMS['duckdb_memory_limit']}'")
con.execute(f"SET temp_directory='{esc(PATHS['full_run_tmp'])}'")
con.execute("SET preserve_insertion_order=false")
con.execute(f"""CREATE TEMP VIEW monthly_base AS
 SELECT permno,"month",mthret,mthretflg,mthprc,market_equity_thousand,
   year("month")*12+month("month") month_index
 FROM read_parquet('{esc(monthly_path)}')""")
con.execute("""CREATE TEMP VIEW windowed AS
 SELECT *,
   mthret r1,
   count(mthret) OVER w12_2 n12_2,
   min(mthret) OVER w12_2 min12_2,
   sum(CASE WHEN mthret=-1 THEN -1e300
            WHEN mthret>-1 THEN ln(1+mthret) END) OVER w12_2 logsum12_2,
   max(CASE WHEN upper(trim(coalesce(mthretflg,'')))
       IN ('MP','MISSING PRICE','P') THEN 1 ELSE 0 END) OVER w12_2
       missing_price_flag12_2,
   max(mthprc) OVER wprice_t13 price_t13,
   market_equity_thousand lag_me,
   count(mthret) OVER w12_7 n12_7,
   min(mthret) OVER w12_7 min12_7,
   sum(CASE WHEN mthret=-1 THEN -1e300
            WHEN mthret>-1 THEN ln(1+mthret) END) OVER w12_7 logsum12_7,
   count(mthret) OVER w36_13 n36_13,
   min(mthret) OVER w36_13 min36_13,
   sum(CASE WHEN mthret=-1 THEN -1e300
            WHEN mthret>-1 THEN ln(1+mthret) END) OVER w36_13 logsum36_13,
   count(mthret) OVER w60_13 n60_13,
   min(mthret) OVER w60_13 min60_13,
   sum(CASE WHEN mthret=-1 THEN -1e300
            WHEN mthret>-1 THEN ln(1+mthret) END) OVER w60_13 logsum60_13
 FROM monthly_base
 WINDOW
   wprice_t13 AS (PARTITION BY permno ORDER BY month_index RANGE BETWEEN 12 PRECEDING AND 12 PRECEDING),
   w12_2 AS (PARTITION BY permno ORDER BY month_index RANGE BETWEEN 11 PRECEDING AND 1 PRECEDING),
   w12_7 AS (PARTITION BY permno ORDER BY month_index RANGE BETWEEN 11 PRECEDING AND 6 PRECEDING),
   w36_13 AS (PARTITION BY permno ORDER BY month_index RANGE BETWEEN 35 PRECEDING AND 12 PRECEDING),
   w60_13 AS (PARTITION BY permno ORDER BY month_index RANGE BETWEEN 59 PRECEDING AND 12 PRECEDING)
""")
feature_query="""SELECT permno,"month",
 r1 r2_1,
 CASE WHEN n12_2=11 AND min12_2>=-1 AND price_t13 IS NOT NULL
       AND lag_me>0 AND missing_price_flag12_2=0
   THEN exp(logsum12_2)-1 END r12_2,
 CASE WHEN n12_7=6 AND min12_7>=-1 THEN exp(logsum12_7)-1 END r12_7,
 CASE WHEN n36_13=24 AND min36_13>=-1 THEN exp(logsum36_13)-1 END r36_13,
 r1 "ST_Rev",
 CASE WHEN n60_13=48 AND min60_13>=-1 THEN exp(logsum60_13)-1 END "LT_Rev",
 (n12_2=11 AND min12_2>=-1 AND price_t13 IS NOT NULL
   AND lag_me>0 AND missing_price_flag12_2=0) r12_2_eligible
 FROM windowed"""
out=output_path(ROOT,RUN_MODE,"16_compute_stock_return_features","data/processed/stock/stock_return_features_raw.parquet")
aud=output_path(ROOT,RUN_MODE,"16_compute_stock_return_features","data/audit/16_r12_2_exclusions.csv")
staged=PATHS["full_run_tmp"]/"16_stock_return_features_staged.parquet"
staged.unlink(missing_ok=True)
duckdb_copy_parquet_atomic(con,f"SELECT * EXCLUDE(r12_2_eligible) FROM ({feature_query})",staged)
duckdb_copy_csv_atomic(con,f"""SELECT permno,"month",'kenneth_french_inclusion_failed' reason_code
 FROM ({feature_query}) WHERE NOT r12_2_eligible""",aud)
con.execute(f"CREATE TEMP VIEW features AS SELECT * FROM read_parquet('{esc(staged)}')")
input_count=int(con.execute("SELECT count(*) FROM monthly_base").fetchone()[0])
feature_count=int(con.execute("SELECT count(*) FROM features").fetchone()[0])
excluded_count=int(con.execute(f"SELECT count(*) FROM ({feature_query}) WHERE NOT r12_2_eligible").fetchone()[0])
duplicate_count=int(con.execute("""SELECT coalesce(sum(n-1),0) FROM (
 SELECT count(*) n FROM features GROUP BY permno,"month" HAVING count(*)>1)""").fetchone()[0])
assert duplicate_count==0
con.close()
out.parent.mkdir(parents=True,exist_ok=True)
os.replace(staged,out)
INPUTS_USED=["stock_monthly_clean.parquet"]; OUTPUTS_WRITTEN=[str(p.relative_to(ROOT)) for p in [out,aud]]
INPUT_COUNTS={"monthly":input_count}; OUTPUT_COUNTS={"stock_month_features":feature_count}; PRIMARY_KEY=["permno","month"]
DUPLICATE_KEY_COUNT=duplicate_count; EXCLUDED_ROW_COUNT=excluded_count; EXCEPTION_FILE=str(aud.relative_to(ROOT)); STAGE_STATUS="passed"
STAGE_NOTES=["All windows use integer calendar-month RANGE frames without gap compression.","CRSP -99.0 and other special codes were converted to missing upstream and never compounded as returns.","Full return features were computed by external-memory DuckDB windows."]


In [ ]:
from src.audit_utils import write_run_summary
summary_path = write_run_summary(
    ROOT, "16_compute_stock_return_features", stage="16", started_at=STARTED_AT,
    input_files=INPUTS_USED, output_files=OUTPUTS_WRITTEN,
    input_row_counts=INPUT_COUNTS, output_row_counts=OUTPUT_COUNTS,
    primary_key=PRIMARY_KEY, duplicate_key_count=DUPLICATE_KEY_COUNT,
    excluded_row_count=EXCLUDED_ROW_COUNT, exception_file=EXCEPTION_FILE,
    status=STAGE_STATUS, notes=STAGE_NOTES,
)
print(summary_path)
